# Introduction to the `mediatools` `ffmpeg` Interface

The `mediatools` package maintains a custom interface for using the ffmpeg library.

In [1]:
import pathlib

import mediatools

Let's download a public video file for demonstration.

In [2]:
import tempfile
import requests

def download_test_video(
    target_path: pathlib.Path,
    url: str = "https://storage.googleapis.com/public_data_09324832787/totk_secret_attack.mkv",
) -> pathlib.Path:
    """Download a test video for the entire test session"""
    
    print(f"\nDownloading test video to {target_path}...")
    response = requests.get(url, stream=True)
    with open(target_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    
    return target_path
    

td = tempfile.TemporaryDirectory()
temp_dir = pathlib.Path(td.name)

ex_vid = download_test_video(temp_dir / "test_video.mkv")
ex_vid

PosixPath('/tmp/tmp270jwicc/test_video.mkv')

## Probe

The ability to probe video metadata is one of the most useful features of `ffmpeg`. The `mediatools` uses the `ProbeInfo` type to store information returned from a probe.

In [3]:
probe = mediatools.ffmpeg.probe(ex_vid)
probe

ProbeInfo(fname='/tmp/tmp270jwicc/test_video.mkv', nb_streams=2, nb_programs=0, format_name='matroska,webm', format_long_name='Matroska / WebM', start_time='0.000000', bit_rate=1480411, dur='23.687000', size=4383314, probe_score=100, tags={'ENCODER': 'Lavf60.16.100'}, video_streams=[VideoStreamInfo(stream_ind=0, codec_name='h264', codec_long_name='H.264 / AVC / MPEG-4 AVC / MPEG-4 part 10', start_time='0.003000', start_pts=3, time_base='1/1000', tags={'ENCODER': 'Lavc60.31.102 libx264', 'DURATION': '00:00:23.669000000'}, disposition={'default': False, 'dub': False, 'original': False, 'comment': False, 'lyrics': False, 'karaoke': False, 'forced': False, 'hearing_impaired': False, 'visual_impaired': False, 'clean_effects': False, 'attached_pic': False, 'timed_thumbnails': False, 'non_diegetic': False, 'captions': False, 'descriptions': False, 'metadata': False, 'dependent': False, 'still_image': False}, height=854, width=480, coded_width=480, coded_height=854, bits_per_raw_sample=8, avg_

### Basic Metadata

The `tags` property maintains metadata that is attached to the video. This could vary according to the camera used or any converter/translation software applied to the video.

In [4]:
probe.tags

{'ENCODER': 'Lavf60.16.100'}

In [5]:
probe.bit_rate, probe.duration, probe.size, probe.probe_score, probe.resolution_str()

(1480411, 23.687, 4383314, 100, '480x854')

### Streams

Each video maintains multiple video or audio streams, depending on the type of video file. The `streams` attribute allows you to explore the streams. Here you can see we have one video stream and one audio stream.

In [6]:
len(probe.video_streams), len(probe.audio_streams), len(probe.other_streams)

(1, 1, 0)

You can get the first video stream (in the order it appears in the original file) using the `video` attribute. This is convenient because most video file types have only one video stream.

In [7]:
probe.video

VideoStreamInfo(stream_ind=0, codec_name='h264', codec_long_name='H.264 / AVC / MPEG-4 AVC / MPEG-4 part 10', start_time='0.003000', start_pts=3, time_base='1/1000', tags={'ENCODER': 'Lavc60.31.102 libx264', 'DURATION': '00:00:23.669000000'}, disposition={'default': False, 'dub': False, 'original': False, 'comment': False, 'lyrics': False, 'karaoke': False, 'forced': False, 'hearing_impaired': False, 'visual_impaired': False, 'clean_effects': False, 'attached_pic': False, 'timed_thumbnails': False, 'non_diegetic': False, 'captions': False, 'descriptions': False, 'metadata': False, 'dependent': False, 'still_image': False}, height=854, width=480, coded_width=480, coded_height=854, bits_per_raw_sample=8, avg_frame_rate='30/1', r_frame_rate='30/1', chroma_location='left', color_range='tv', color_space='bt709', field_order='progressive', has_b_frames=2, closed_captions=False, is_avc='true', level=31, pix_fmt='yuv420p', profile='High', refs=1)

In [8]:
probe.video.resolution, probe.video.codec_name, probe.video.start_time, probe.video.frame_rate

((480, 854), 'h264', '0.003000', 30)

Similarly, the first audio channel can be retrieved using `audio` attribute.

In [9]:
probe.audio

AudioStreamInfo(stream_ind=1, codec_name='vorbis', codec_long_name='Vorbis', start_time='0.000000', start_pts=0, time_base='1/1000', tags={'VARIANT_BITRATE': '0', 'ID3V2_PRIV.COM.APPLE.STREAMING.TRANSPORTSTREAMTIMESTAMP': '\\x00\\x00\\x00\\x00\\x00\\x00\\x00\\x00', 'ENCODER': 'Lavc60.31.102 libvorbis', 'DURATION': '00:00:23.687000000'}, disposition={'default': False, 'dub': False, 'original': False, 'comment': False, 'lyrics': False, 'karaoke': False, 'forced': False, 'hearing_impaired': False, 'visual_impaired': False, 'clean_effects': False, 'attached_pic': False, 'timed_thumbnails': False, 'non_diegetic': False, 'captions': False, 'descriptions': False, 'metadata': False, 'dependent': False, 'still_image': False}, sample_fmt='fltp', sample_rate=44100, channels=2, channel_layout='stereo')

In [10]:
probe.audio.codec_name, probe.audio.start_time, probe.audio.channels, probe.audio.channel_layout, probe.audio.sample_rate

('vorbis', '0.000000', 2, 'stereo', 44100)

## `FFMPEG` Interface

The low-level ffmpeg interface allows you to access the full range of ffmpeg features through a modern interface. Create the command object using the `FFMPEG` default constructor along with the `ffinput` and `ffoutput` functions.

For the first example, I will execute a command that will cut a video halfway through. You would instantiate the command as follows:

In [20]:
output_path = temp_dir / "test_video_output.mp4"
half_duration = probe.duration / 2
print(f'{half_duration=}')

cmd = mediatools.ffmpeg.FFMPEG(
    inputs = [
        mediatools.ffmpeg.ffinput(str(ex_vid), ss=probe.start_time, to=half_duration),
    ],
    outputs = [
        mediatools.ffmpeg.ffoutput(str(output_path), y=True),
    ],
)
cmd

half_duration=11.8435


FFMPEG(inputs=[FFInput(path='/tmp/tmp270jwicc/test_video.mkv', args=FFInputArgs(f=None, t=None, ss='0.000000', to=11.8435, itsoffset=None, c_v=None, c_a=None, r=None, s=None, pix_fmt=None, aspect=None, vframes=None, top=None, ar=None, ac=None, aframes=None, vol=None, hwaccel=None, hwaccel_device=None, map_metadata=None, map_chapters=None, probesize=None, analyzeduration=None, fpsprobesize=None, safe=None, loop=None, stream_loop=None, accurate_seek=None, seek_timestamp=None, other_args=None, other_flags=None))], outputs=[FFOutput(path='/tmp/tmp270jwicc/test_video_output.mp4', args=FFOutputArgs(y=True, maps=[], map_metadata=None, map_chapters=None, ss=None, t=None, duration=None, to=None, c_v=None, b_v=None, crf=None, q_v=None, maxrate=None, bufsize=None, framerate=None, fps=None, s=None, aspect=None, pix_fmt=None, vframes=None, keyint_min=None, g=None, bf=None, profile_v=None, level=None, tune=None, c_a=None, b_a=None, ar=None, ac=None, vol=None, aframes=None, profile_a=None, q_a=None, 

The `run` the command will actually execute the ffmpeg call and return a `FFMPEGResult` instance with information about the command and `stderr`/`stdout`.

In [12]:
result = cmd.run()
result

FFMPEGResult(command=ffmpeg -ss 0.000000 -to 11.8435 -i /tmp/tmp270jwicc/test_video.mkv -hide_banner -nostats -y /tmp/tmp270jwicc/test_video_output.mp4, returncode=0, output_length=3965)

In [17]:
print(result.stderr)

Input #0, matroska,webm, from '/tmp/tmp270jwicc/test_video.mkv':
  Metadata:
    ENCODER         : Lavf60.16.100
  Duration: 00:00:23.69, start: 0.000000, bitrate: 1480 kb/s
  Stream #0:0: Video: h264 (High), yuv420p(tv, bt709, progressive), 480x854, 30 fps, 30 tbr, 1k tbn
    Metadata:
      ENCODER         : Lavc60.31.102 libx264
      DURATION        : 00:00:23.669000000
  Stream #0:1: Audio: vorbis, 44100 Hz, stereo, fltp
    Metadata:
      VARIANT_BITRATE : 0
      ID3V2_PRIV.COM.APPLE.STREAMING.TRANSPORTSTREAMTIMESTAMP: \x00\x00\x00\x00\x00\x00\x00\x00
      ENCODER         : Lavc60.31.102 libvorbis
      DURATION        : 00:00:23.687000000
Stream mapping:
  Stream #0:0 -> #0:0 (h264 (native) -> h264 (libx264))
  Stream #0:1 -> #0:1 (vorbis (native) -> aac (native))
Press [q] to stop, [?] for help
[libx264 @ 0x5f67ec190000] using cpu capabilities: MMX2 SSE2Fast SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2
[libx264 @ 0x5f67ec190000] profile High, level 3.1, 4:2:0, 8-bit
[libx264 @ 0x5f67ec19

You can verify that the command was successful by probing the new file. Notice the difference between the requested duration and the actual duration. This is an issue because videos are composed of discrete frames.

In [19]:
mediatools.ffmpeg.probe(output_path).duration

11.866667